# 12.a — Ad hoc test: the effect of the AMOC collapse against the internal variability of the runs

**Ad hoc test, outside the suitability chain**: it reads the outputs of 0.1, 0.5, 1.1.d, 1.3.a, 2.g and 3.g and the
daily climate of the runs, and writes only in `<out_root>/test_internal_variability/<route>/<set>/`.

**Question.** Each run is one member of 50 model years of a stabilised climate, and its BIO variables are means over
its 49 seasons. Another 49 seasons of the same climate would give slightly different means, hence a slightly
different suitability, above all in the cells close to the boundary of the SVM or of the ECOCROP TMIN. Are the effects
of the collapse of 6.b (today's crop area: +2 to +17 points; suitable land: −2 to −6 % for maize, soybean and rice)
larger than this internal variability? With one member per run, the variability is estimated inside each run.

**Method.**
1. The harvest years of each run (those of 1.3.a: from the second year of the file to the last) are split into two
   halves, **H1** (the first 24) and **H2** (the last 25). The BIO variables of each half and of the whole run
   (**full**) are computed with the code of 1.3.a (same calendar, same definitions) and turned into delta BIO variables
   on ERA5 with the change against EC-Earth3 historical 1981–2014 (the raw historical BIO of 1.3.a: the same reference
   for both halves, so its own sampling cancels in hosing − no hosing).
2. The final rule of 4.g and 6.b, unchanged: SVM of 3.g at equal class priors, the training cells predicted by the fold
   model that did not see them (cross-fitting), novel climate (MESS < 0, NT2 > 1 or kernel support < 0.01) not
   suitable, probability ≥ 0.5 and growing-season temperature of the lower part of the cell ≥ the ECOCROP TMIN, soil
   rating ≥ 10, low and high input. Check: **full** reproduces the delta BIO of 1.3.a and the maps of 6.b.
3. **Effect of the collapse** (hosing − no hosing) in full, H1 and H2, for the metrics of 6.b: the change of today's
   crop area suitable (points of the MIRCA2000 area of the crop), and the change of the suitable land (%, cells
   assessable in both runs), globally and in the regions of 6.b.

**Reading.**
- **The two halves agree**: the effect has the same sign in H1 and in H2 (two independent estimates of 24–25 seasons).
- **Effect against noise**: in a run the two halves differ by d = X(H1) − X(H2), the sampling noise of 24–25 seasons
  (variance 2σ²). The full effect, a difference of two 49-season estimates, has the standard error SE = σ =
  √(mean d² / 2), d pooled over the four runs. |effect| > 2 SE: beyond the internal variability at about 95 %.
  A drift inside a run (an ocean still adjusting after the hosing or the stabilisation of the CO₂) also enters d,
  so the noise is if anything overestimated: the test is conservative. With four differences per metric the SE is
  itself uncertain: the agreement of the halves is the second criterion.
- **Flips**: the share of today's crop area that changes class between the two halves of the same run, against the
  share that changes between hosing and no hosing (full).

Limits: one climate model and one member per run; halves of 24–25 seasons (the noise of 49 seasons is derived, not
observed); the noise of the reference (EC-Earth3 historical, ERA5) is common to every run and not tested.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: the final set of 1.3.a, 4.g and 6.b
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = "no_rad"              # the runs have no rsds yet
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")     # level of the delta and training climate (1.1.d)
HIST = "gcm_historical"                               # reference of the change: raw BIO of 1.3.a (1981-2014)
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
RUN_DATA = {}                     # run -> (dataset, (first, last year)); default the whole file of the run
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVELS = ["low", "high"]          # GAEZ input level of the soil rating (2.g)
PR_MIN = 0.1                      # mm/day: below, the precipitation change is additive (1.3.a)
KERNEL_EPS = 0.01                 # novel where the kernel support < KERNEL_EPS (4.g)
N_JOBS = min(5, os.cpu_count() or 1)   # fold models (only when 4.g has not saved them)
OVERWRITE = False                 # False: the delta BIO of the halves already written are read back
PARTS = ["full", "H1", "H2"]
assert CHOSEN_RAD == "no_rad", "no_rad only (no rsds in the runs)"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
RAW = cu.out_dir(CFG, "1_biovariables", f"GCM_raw_{ROUTE}", SET)        # 1.3.a: raw BIO of EC-Earth3 historical
DELTA13 = cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET)  # 1.3.a: delta BIO of the whole runs (check)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
AMOC_DIR = cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, SET)            # 6.b (check)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
OUT = cu.out_dir(CFG, "test_internal_variability", ROUTE, SET)          # the only folder written
BIO_OUT = OUT / "bio"; BIO_OUT.mkdir(exist_ok=True)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
I, J = np.nonzero(LAND)
LAT_RAD = np.radians(GRID["lat"][I])
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
DT = cu.tmin_warming(CFG)
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,                                                   # as 6.b
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
era5_file = lambda crop: ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
half_file = lambda run, part, crop: BIO_OUT / f"BIOvariables_delta_{run}_{part}_{crop}.nc"
print(f"set {SET} | runs {RUNS} | parts {PARTS} | written only in {OUT}")

## 1. Delta BIO variables of the whole runs and of their two halves (code of 1.3.a)

In [ ]:
def compute_Ra(lat_rad, doy):
    Gsc = 0.0820
    dr = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)
    ws = np.arccos(np.clip(-np.tan(lat_rad) * np.tan(delta), -1, 1))
    return (24 * 60 / np.pi) * Gsc * dr * (ws * np.sin(lat_rad) * np.sin(delta) + np.cos(lat_rad) * np.cos(delta) * np.sin(ws))


def eto_day(tx, tn, lat_rad, doy):
    # compute_ETo_day of 1.1.d / 1.3.a (Hargreaves, Ra in mm/day equivalent)
    tm = (tx + tn) / 2.0
    return 0.0023 * compute_Ra(lat_rad, doy) / (2.501 - 0.002361 * tm) * (tm + 17.8) * np.sqrt(np.maximum(tx - tn, 0.0))


def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def bio_of(TMAX, TMIN, PR, ETO, dates, sow, ant, mat, harvest_years):
    # compute_bio_variables of 1.3.a on (days, cells) arrays, without radiation (no_rad): the BIO variables averaged
    # over the given harvest years (seasons crossing New Year start in hy - 1)
    n = sow.size
    yr_all, doy_all, mon_all = dates.year.values, dates.dayofyear.values, dates.month.values
    keys = (1, 2, 4, 5, 6, 7, 9, 10, 11, 12)
    s_, c_ = {k: np.zeros(n) for k in keys}, {k: np.zeros(n) for k in keys}
    valid = ~np.isnan(sow) & ~np.isnan(mat)
    wrap = valid & (mat > 365)
    present = set(yr_all)

    def acc(k, v):
        f = np.isfinite(v); s_[k] += np.where(f, v, 0); c_[k] += f

    for hy in harvest_years:
        if not (yr_all == hy).any():
            continue
        idx = np.where((yr_all == hy) | (yr_all == hy - 1))[0]
        yr3, doy3, mon3 = yr_all[idx][:, None], doy_all[idx][:, None], mon_all[idx][:, None]
        t3 = doy3 + np.where(yr3 == hy, ylen(hy - 1), 0)
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s3, a3, m3 = np.nan_to_num(sow)[None] + shift, np.nan_to_num(ant)[None] + shift, np.nan_to_num(mat)[None] + shift
        vp = valid[None] & (((hy - 1) in present) | ~wrap[None])
        p1 = (t3 >= s3) & (t3 < a3)
        p2 = (t3 >= a3) & (t3 <= m3)
        gs = (p1 | p2) & vp; p1 &= vp; p2 &= vp
        tm, tn, pr, et = TMAX[idx], TMIN[idx], PR[idx], ETO[idx]
        pm = lambda a, m: np.nanmean(np.where(m, a, np.nan), axis=0)
        acc(1, pm(tm, p1)); acc(2, pm(tn, p1)); acc(4, pm(et, p1)); acc(5, pm(pr, p1))
        acc(6, pm(tm, p2)); acc(7, pm(tn, p2)); acc(9, pm(et, p2)); acc(10, pm(pr, p2))
        tmx, tmn = np.full(n, np.nan), np.full(n, np.nan)
        for m in range(1, 13):
            mm = gs & (mon3 == m)
            if not mm.any():
                continue
            tmx = np.fmax(tmx, np.nanmean(np.where(mm, tm, np.nan), axis=0))
            tmn = np.fmin(tmn, np.nanmean(np.where(mm, tn, np.nan), axis=0))
        acc(11, tmx); acc(12, tmn)
    out = {}
    for k in keys:
        with np.errstate(invalid="ignore", divide="ignore"):
            v = np.where(c_[k] > 0, s_[k] / c_[k], np.nan)
        v[~valid] = np.nan
        out[f"BIO{k:02d}"] = v
    return out


def grid_of(x):
    g = np.full(LAND.shape, np.nan, np.float32)
    g[I, J] = x
    return g


def delta(e5, h, raw, cols, pr_cols):
    # 1.3.a: ERA5 + (run - historical), ratio for precipitation (additive where historical < PR_MIN), >= 0
    out = {}
    for c in cols:
        r = raw[c]
        if c in pr_cols:
            v = np.where(h[c] >= PR_MIN, e5[c] * r / np.where(h[c] >= PR_MIN, h[c], 1.0), e5[c] + (r - h[c]))
        else:
            v = e5[c] + (r - h[c])
        out[c] = np.clip(v, 0, None) if c in pr_cols else v
    return out


with xr.open_dataset(era5_file(CROPS[0])) as d0:
    COLS = [v for v in d0.data_vars if v.startswith("BIO") and not v.endswith("e") and v not in ("BIO03", "BIO08")]
PR_COLS = [c for c in COLS if c in ("BIO05", "BIO10")]
E5, HB, CAL = {}, {}, {}
for crop in CROPS:
    with xr.open_dataset(era5_file(crop)) as e:
        E5[crop] = {c: e[c].values for c in COLS}
        CAL[crop] = {v: e[v].values for v in ("sowing_doy", "phase2_start_doy", "maturity_doy")}
    with xr.open_dataset(RAW / f"BIOvariables_raw_{HIST}_{crop}.nc") as h:
        HB[crop] = {c: h[c].values for c in COLS}


def parts_of(dates):
    years = np.unique(dates.year)
    hy = np.arange(years[0] + 1, years[-1] + 1)          # harvest years of 1.3.a
    k = hy.size // 2
    return {"full": hy, "H1": hy[:k], "H2": hy[k:]}


HARVEST = {}
for run in RUNS:
    if not OVERWRITE and all(half_file(run, p, c).exists() for p in PARTS for c in CROPS):
        with xr.open_dataset(half_file(run, "H1", CROPS[0])) as d:
            HARVEST[run] = {p: d.attrs.get(f"harvest_years_{p}") for p in PARTS}
        print(f"[already done] {run}: {HARVEST[run]}"); continue
    t0 = time.time()
    name, period = RUN_DATA.get(run, (run, None))
    das = {v: cu.read_climate(CFG, name, v, GRID, period) for v in ("tasmax", "tasmin", "pr")}
    dates = pd.DatetimeIndex(das["tasmax"]["time"].values).normalize()
    TMAX, TMIN, PR = (cu.land_series(das[v], (I, J)) for v in ("tasmax", "tasmin", "pr"))
    doys = dates.dayofyear.values
    ETO = np.empty_like(TMAX)
    for t in range(dates.size):
        ETO[t] = eto_day(TMAX[t], TMIN[t], LAT_RAD, doys[t])
    P = parts_of(dates)
    HARVEST[run] = {p: f"{P[p][0]}-{P[p][-1]} ({P[p].size})" if P[p].size else "none" for p in PARTS}
    print(f"{run}: {dates[0].date()} -> {dates[-1].date()}, read in {(time.time() - t0) / 60:.1f} min | {HARVEST[run]}", flush=True)
    for crop in CROPS:
        sow, ant, mat = (CAL[crop][v][I, J].astype(float) for v in ("sowing_doy", "phase2_start_doy", "maturity_doy"))
        for p in PARTS:
            b = bio_of(TMAX, TMIN, PR, ETO, dates, sow, ant, mat, P[p])
            raw = {c: grid_of(b[c]) for c in COLS}
            dv = delta(E5[crop], HB[crop], raw, COLS, PR_COLS)
            ds = xr.Dataset({c: (("lat", "lon"), dv[c].astype("float32")) for c in COLS},
                            coords={"lat": GRID["lat"], "lon": GRID["lon"]},
                            attrs={"run": run, "part": p, **{f"harvest_years_{q}": HARVEST[run][q] for q in PARTS},
                                   "description": f"delta BIO: ERA5 + ({run} {p} - {HIST}), code of 1.3.a, {SET}"})
            for v in CAL[crop]:
                ds[v] = (("lat", "lon"), CAL[crop][v])
            ds.to_netcdf(half_file(run, p, crop))
        print(f"  {crop} done", flush=True)
    del TMAX, TMIN, PR, ETO
    print(f"{run}: {(time.time() - t0) / 60:.1f} min", flush=True)

# check: the full run reproduces the delta BIO of 1.3.a
rows = []
for run in RUNS:
    for crop in CROPS:
        f13 = DELTA13 / f"BIOvariables_delta_{run}_{crop}.nc"
        if not f13.exists():
            continue
        with xr.open_dataset(f13) as a, xr.open_dataset(half_file(run, "full", crop)) as b:
            rows.append({"run": run, "crop": crop, **{c: float(np.nanmax(np.abs(a[c].values - b[c].values)[LAND])) for c in COLS}})
CHECK_BIO = pd.DataFrame(rows)
if len(CHECK_BIO):
    print("max |full - 1.3.a| over the land cells (should be ~0):")
    display(CHECK_BIO.set_index(["run", "crop"]).max().to_frame("max").T)
else:
    print("no delta BIO of 1.3.a for these runs: check skipped")

## 2. Final suitability of every part (rule of 4.g and 6.b)

ERA5 and, for each run, full, H1 and H2. Check: the full parts against the maps of 6.b.

In [ ]:
def suitability(crop):
    # {"ERA5" or (run, part): {"suit": {level: bool (lat, lon)}, "novel": bool}}, as 4.g + 6.b
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    cols, n1, n0 = bundle["bio_cols"], bundle["n_class1"], bundle["n_class0"]
    assert set(cols) <= set(COLS), (cols, COLS)
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(era5_file(crop)) as ds:
        REF = cu.bio_array(ds, cols, CFG)[ii, jj]
    REF = REF[np.isfinite(REF).all(axis=1)]
    kfun = cu.kernel_support(bundle["model"])
    Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[cols].values.astype(float),
                            tr["label"].values.astype(int), bundle, n_jobs=N_JOBS)
    ssr = {lv: np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0) for lv in LEVELS}
    which = cu.ECOCROP_TMIN[crop][1]
    files = {"ERA5": era5_file(crop), **{(r, p): half_file(r, p, crop) for r in RUNS for p in PARTS}}
    out = {}
    for key, path in files.items():
        with xr.open_dataset(path) as ds:
            A = cu.bio_array(ds, cols, CFG)
            tg = np.asarray(cu.growing_temperature(ds, which), float)
        X = A.reshape(-1, len(cols))
        ok = np.isfinite(X).all(axis=1) & LAND.ravel()
        p, nov = np.full(X.shape[0], np.nan), np.zeros(X.shape[0], bool)
        if ok.any():                                              # (a part without any season: all NaN)
            Xs = bundle["scaler"].transform(X[ok])
            p[ok] = bundle["model"].predict_proba(Xs)[:, 1]
            nov[ok] = (cu.mess(X[ok], REF)[0] < 0) | (cu.nt2(X[ok], REF) > 1) | (kfun(Xs) < KERNEL_EPS)
        p, nov = p.reshape(LAND.shape), nov.reshape(LAND.shape)
        g = cu.crossfit_proba(Fd, A[ii, jj])                      # training cells: the fold model that did not see them
        fin = np.isfinite(g)
        p[ii[fin], jj[fin]] = g[fin]
        p = cu.balanced_proba(p, n1, n0)
        nov &= np.isfinite(p)
        proba = np.where(nov, 0.0, p)
        clim = cu.climate_suitable(proba, tg, crop, DT) & np.isfinite(proba)
        out[key] = {"suit": {lv: LAND & cu.suitable(clim, ssr[lv]) for lv in LEVELS}, "novel": nov}
    return out


S = {}
for crop in CROPS:
    t0 = time.time()
    S[crop] = suitability(crop)
    print(f"{crop}: {len(S[crop])} climates in {(time.time() - t0) / 60:.1f} min", flush=True)

rows = []                                       # check: full against 6.b
for crop in CROPS:
    f6 = AMOC_DIR / f"climate_soil_scenarios_{crop}.nc"
    if not f6.exists():
        continue
    with xr.open_dataset(f6) as ds:
        for run in [r for r in RUNS if r in ds.scenario.values]:
            for lv in LEVELS:
                a = ds[f"suitable_{lv}"].sel(scenario=run).values == 1
                b = S[crop][(run, "full")]["suit"][lv]
                rows.append({"crop": crop, "run": run, "level": lv, "cells differing": int((a != b)[LAND].sum()),
                             "of land cells (%)": 100 * (a != b)[LAND].mean()})
CHECK_6B = pd.DataFrame(rows)
if len(CHECK_6B):
    print("full against 6.b (should be 0):")
    display(CHECK_6B.groupby("level")[["cells differing", "of land cells (%)"]].max())
else:
    print("no 6.b output for these runs: check skipped")

## 3. Effect of the collapse in each part, noise between the halves, flips

Metrics of 6.b: `today's crop area` = change of the share of the MIRCA2000 area of the crop in suitable cells
(points); `<region> land` = change of the suitable land, cells assessable in both runs (%). Effect = hosing − no
hosing; noise d = H1 − H2 of each run (for the land, the change of H1 against H2 in %); SE of the full effect =
√(mean d² / 2), d of the four runs pooled.

In [ ]:
mha = lambda m: float(np.nansum(LAND_MHA * m))
pct = lambda a, b: 100 * (a - b) / b if b > 0.5 else np.nan        # < 0.5 Mha: not reported (as 6.b)
rows, noise, flips = [], [], []
for crop in CROPS:
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    R = S[crop]
    for lv in LEVELS:
        def metrics(a, b, na, nb):
            # change from climate b to climate a: today's crop area (points), suitable land per region (%)
            both = LAND & ~(na | nb)
            m = {"today's crop area (points)": share(a) - share(b)}
            for reg, rm in REGIONS.items():
                m[f"{reg} land (%)"] = pct(mha(a & both & rm), mha(b & both & rm))
            return m
        for h, o in PAIRS:
            for p in PARTS:
                A, B = R[(h, p)], R[(o, p)]
                for k, v in metrics(A["suit"][lv], B["suit"][lv], A["novel"], B["novel"]).items():
                    rows.append({"crop": crop, "level": lv, "pair": f"{h} - {o}", "metric": k, "part": p, "effect": v})
            A, B = R[(h, "full")], R[(o, "full")]
            ok = ~(A["novel"] | B["novel"])
            flips.append({"crop": crop, "level": lv, "comparison": f"{h} vs {o} (full)",
                          "today's crop area flipped (%)": share((A["suit"][lv] != B["suit"][lv]) & ok),
                          "land flipped (Mha)": mha((A["suit"][lv] != B["suit"][lv]) & ok & LAND)})
        for run in RUNS:
            A, B = R[(run, "H1")], R[(run, "H2")]
            for k, v in metrics(A["suit"][lv], B["suit"][lv], A["novel"], B["novel"]).items():
                noise.append({"crop": crop, "level": lv, "run": run, "metric": k, "d (H1 - H2)": v})
            ok = ~(A["novel"] | B["novel"])
            flips.append({"crop": crop, "level": lv, "comparison": f"{run}: H1 vs H2",
                          "today's crop area flipped (%)": share((A["suit"][lv] != B["suit"][lv]) & ok),
                          "land flipped (Mha)": mha((A["suit"][lv] != B["suit"][lv]) & ok & LAND)})
EFF = pd.DataFrame(rows).pivot_table(index=["crop", "level", "pair", "metric"], columns="part", values="effect", sort=False)[PARTS]
NOISE = pd.DataFrame(noise)
SE = NOISE.groupby(["crop", "level", "metric"], sort=False)["d (H1 - H2)"].apply(lambda d: np.sqrt(np.nanmean(d ** 2) / 2))
FLIPS = pd.DataFrame(flips).set_index(["crop", "level", "comparison"])
SUM = EFF.join(SE.rename("SE (noise)"), on=["crop", "level", "metric"])
SUM["|full| / SE"] = SUM["full"].abs() / SUM["SE (noise)"]
SUM["same sign in H1 and H2"] = np.sign(SUM["H1"]) == np.sign(SUM["H2"])
SUM["robust"] = SUM["same sign in H1 and H2"] & (SUM["|full| / SE"] > 2)
EFF.to_csv(OUT / "effect_by_part.csv"); NOISE.to_csv(OUT / "noise_halves.csv", index=False)
SUM.to_csv(OUT / "effect_vs_noise.csv"); FLIPS.to_csv(OUT / "flips.csv")
pd.DataFrame(HARVEST).T.to_csv(OUT / "harvest_years.csv")
with pd.option_context("display.max_rows", 400, "display.width", 250):
    show = SUM.reset_index()
    show = show[show.metric.isin(["today's crop area (points)", "global land (%)", "north of 50 N land (%)", "Europe land (%)"])]
    for lv in LEVELS[::-1]:
        print(f"Effect of the collapse against the internal variability, {lv} input (robust: same sign in both halves and |full| > 2 SE)")
        display(show[show.level == lv].drop(columns="level").set_index(["crop", "pair", "metric"]).round(2))
    print("Flips: share of today's crop area changing class (cells assessable in both)")
    display(FLIPS.xs("high", level="level").round(2))

## 4. Figure: effect of the collapse (full, ± 2 SE) and in each half, high input

In [ ]:
LV = "high" if "high" in LEVELS else LEVELS[0]
mets = ["today's crop area (points)", "global land (%)", "north of 50 N land (%)"]
fig, axes = plt.subplots(1, len(mets), figsize=(6.5 * len(mets), 0.55 * len(CROPS) * len(PAIRS) + 1.5), squeeze=False)
for ax, met in zip(axes[0], mets):
    labels, y = [], 0
    for crop in CROPS:
        for h, o in PAIRS:
            key = (crop, LV, f"{h} - {o}", met)
            if key not in SUM.index:
                continue
            r = SUM.loc[key]
            col = "#2166ac" if r["full"] > 0 else "#b2182b"
            ax.barh(y, r["full"], color=col, alpha=0.35 if not r["robust"] else 0.85)
            if np.isfinite(r["SE (noise)"]):
                ax.errorbar(r["full"], y, xerr=2 * r["SE (noise)"], color="k", capsize=3, lw=1)
            ax.plot([r["H1"]], [y], "o", color="#ff7f0e", ms=5); ax.plot([r["H2"]], [y], "s", color="#2ca02c", ms=5)
            labels.append(f"{crop.replace('_rf', '')} {h.split('_')[1] if '_' in h else h}"); y += 1
    ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3); ax.set_title(f"{met}, hosing - no hosing", fontsize=10)
axes[0, 0].plot([], [], "o", color="#ff7f0e", label="first half (H1)"); axes[0, 0].plot([], [], "s", color="#2ca02c", label="second half (H2)")
axes[0, 0].legend(fontsize=8, loc="lower right")
fig.suptitle(f"Effect of the AMOC collapse ({LV} input): full run with ± 2 SE of the internal variability (dark bar: robust)", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / f"effect_vs_noise_{LV}.png", dpi=100, bbox_inches="tight"); plt.show()
print("written:", sorted(p.name for p in OUT.iterdir()))